# MPT Agent System - Gemini Colab Runner

Runtime: Standard (CPU). No GPU required.

Pipeline: TrendScout -> ScriptWriter (Gemini) -> VideoProducer (MPT + Pexels) -> SEO (Gemini). Publisher step is optional and skipped by default.

## Step 1 - Install dependencies and clone repos

In [ ]:
import os

print('Installing system packages (imagemagick, ffmpeg)...')
!apt-get install -y -q imagemagick ffmpeg > /dev/null 2>&1
# Relax ImageMagick policy for Colab
!sed -i 's/rights="none" pattern="PDF"/rights="read|write" pattern="PDF"/g' /etc/ImageMagick-6/policy.xml 2>/dev/null || true

# Clone MoneyPrinterTurbo
if not os.path.exists('/content/MoneyPrinterTurbo'):
    !git clone --depth 1 https://github.com/harry0703/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo
else:
    !cd /content/MoneyPrinterTurbo && git pull --quiet
print('MoneyPrinterTurbo ready')

# Install MPT Python dependencies
!pip install -q -r /content/MoneyPrinterTurbo/requirements.txt 2>&1 | tail -5
!pip install -q loguru edge-tts moviepy imageio-ffmpeg toml 2>&1 | tail -3

# Clone agent system
if not os.path.exists('/content/mpt-agent-system'):
    !git clone --depth 1 https://github.com/Bhushan-Khachane/mpt-agent-system.git /content/mpt-agent-system
else:
    !cd /content/mpt-agent-system && git pull

# Install agent dependencies
!pip install -q feedparser pytrends requests google-api-python-client google-auth-httplib2 google-auth-oauthlib 2>&1 | tail -3

print('All dependencies installed.')

## Step 2 - Configure environment (API keys)

- Gemini API key (starts with AIza): https://aistudio.google.com/app/apikey
- Pexels API key: https://www.pexels.com/api/
- Pixabay API key (optional): https://pixabay.com/api/docs/

In [ ]:
import os, sys, pathlib

# EDIT THESE VALUES
GEMINI_API_KEY  = 'AIza...'          # required
PEXELS_API_KEY  = 'YOUR_PEXELS_KEY'  # required
PIXABAY_API_KEY = ''                 # optional

if not GEMINI_API_KEY.startswith('AIza') or len(GEMINI_API_KEY) < 30:
    raise ValueError('Invalid Gemini key. Get one from https://aistudio.google.com/app/apikey')
if PEXELS_API_KEY in ('YOUR_PEXELS_KEY', '', None):
    raise ValueError('Invalid Pexels key. Get one from https://www.pexels.com/api/')

os.environ['GEMINI_API_KEY']   = GEMINI_API_KEY
os.environ['GEMINI_MODEL']     = 'gemini-2.0-flash'
os.environ['PEXELS_API_KEY']   = PEXELS_API_KEY
os.environ['PIXABAY_API_KEY']  = PIXABAY_API_KEY
os.environ['MPT_DIR']          = '/content/MoneyPrinterTurbo'

AGENT_ROOT = '/content/mpt-agent-system'
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

for d in ['data', 'videos', 'logs', 'credentials']:
    pathlib.Path(f'{AGENT_ROOT}/{d}').mkdir(parents=True, exist_ok=True)
for fp, default in [
    (f'{AGENT_ROOT}/data/topics_queue.json', '[]'),
    (f'{AGENT_ROOT}/data/seen_topics.json',  '[]'),
]:
    p = pathlib.Path(fp)
    if not p.exists():
        p.write_text(default)

print('Config OK')
print('Gemini key prefix:', GEMINI_API_KEY[:12] + '...')
print('Pexels key prefix:', PEXELS_API_KEY[:10] + '...')

## Step 2b - Sanity check (run once before Step 3)

In [ ]:
import os, pathlib, subprocess, requests

ok = True
checks = {
    'Agent root exists' : pathlib.Path('/content/mpt-agent-system').exists(),
    'MPT root exists'   : pathlib.Path('/content/MoneyPrinterTurbo').exists(),
    'MPT cli.py exists' : pathlib.Path('/content/MoneyPrinterTurbo/cli.py').exists(),
    'Queue file exists' : pathlib.Path('/content/mpt-agent-system/data/topics_queue.json').exists(),
    'Gemini key set'    : os.environ.get('GEMINI_API_KEY','').startswith('AIza'),
    'Pexels key set'    : bool(os.environ.get('PEXELS_API_KEY','')),
    'convert in PATH'   : bool(subprocess.run(['which','convert'],capture_output=True).stdout),
    'ffmpeg in PATH'    : bool(subprocess.run(['which','ffmpeg'],capture_output=True).stdout),
}
for name, passed in checks.items():
    print(('OK   ' if passed else 'FAIL ') + name)
    if not passed: ok = False

print('
Testing Gemini API...')
try:
    key   = os.environ['GEMINI_API_KEY']
    model = os.environ['GEMINI_MODEL']
    resp = requests.post(
        f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={key}',
        json={'contents':[{'parts':[{'text':'Say: GEMINI OK'}]}]},
        timeout=15
    )
    resp.raise_for_status()
    text = resp.json()['candidates'][0]['content']['parts'][0]['text'].strip()
    print('Gemini response:', text)
except Exception as e:
    print('Gemini test failed:', e)
    ok = False

print('
All checks passed.' if ok else '
Some checks failed. Fix before continuing.')

## Step 3 - Run TrendScout

In [ ]:
import importlib, sys, json, pathlib

if 'agents.trend_scout' in sys.modules:
    importlib.reload(sys.modules['agents.trend_scout'])
from agents.trend_scout import run_trend_scout

topics = run_trend_scout()
qf      = pathlib.Path('/content/mpt-agent-system/data/topics_queue.json')
queue   = json.loads(qf.read_text())
pending = [t for t in queue if t.get('status') == 'pending']
print('Queue file:', qf)
print('Total topics in queue:', len(queue))
print('Pending topics:', len(pending))
print('
Sample topics:')
for t in pending[:5]:
    print('-', t['niche'], '|', t['title'][:80])

## Step 4 - Run ScriptWriter (Gemini)

In [ ]:
import importlib, sys, json, pathlib

if 'agents.script_writer' in sys.modules:
    importlib.reload(sys.modules['agents.script_writer'])
if 'agents.llm_client' in sys.modules:
    importlib.reload(sys.modules['agents.llm_client'])
from agents.script_writer import run_script_writer

n = run_script_writer(batch_size=6)
queue    = json.loads(pathlib.Path('/content/mpt-agent-system/data/topics_queue.json').read_text())
scripted = [t for t in queue if t.get('status') == 'scripted']
failed   = [t for t in queue if t.get('status') == 'script_failed']
print('Scripts written this run:', n)
print('Total scripted topics in queue:', len(scripted))
print('Script failures:', len(failed))
if scripted:
    t0 = scripted[0]
    print('
Sample scripted topic:')
    print('Niche :', t0['niche'])
    print('Title :', t0['title'])
    print('Script:
', t0['script'])

## Step 5 - Run VideoProducer (MoneyPrinterTurbo CLI)

In [ ]:
import importlib, sys

if 'agents.video_producer' in sys.modules:
    importlib.reload(sys.modules['agents.video_producer'])
from agents.video_producer import run_video_producer

run_video_producer(batch_size=1)  # increase after first video works

## Step 6 - Run SEO Agent (Gemini)

In [ ]:
import importlib, sys
if 'agents.seo_agent' in sys.modules:
    importlib.reload(sys.modules['agents.seo_agent'])
from agents.seo_agent import run_seo_agent
run_seo_agent(batch_size=6)

## Step 7 - Publisher (skipped by default)

In [ ]:
# Publisher is optional. You preferred manual upload, so we do not call it here.
# If you want automatic YouTube upload, uncomment the last line.
import importlib, sys
if 'agents.publisher' in sys.modules:
    importlib.reload(sys.modules['agents.publisher'])
from agents.publisher import run_publisher

print('Publisher step skipped. Videos (and metadata) will be queued for manual upload.')
# run_publisher(batch_size=3)

## Step 8 - Check pipeline status

In [ ]:
import json, pathlib
from collections import Counter

qf     = pathlib.Path('/content/mpt-agent-system/data/topics_queue.json')
queue  = json.loads(qf.read_text())
counts = Counter(t.get('status','unknown') for t in queue)
print('Queue file:', qf)
print('Total items:', len(queue))
print('
Status breakdown:')
for status, n in sorted(counts.items()):
    print(f'  {status:25s}: {n}')

videos = list(pathlib.Path('/content/mpt-agent-system/videos').glob('*.mp4'))
if videos:
    print('
Videos generated:')
    for v in videos:
        size_mb = v.stat().st_size / 1024 / 1024
        print(f'  {v.name} ({size_mb:.1f} MB)')

manual_q = pathlib.Path('/content/mpt-agent-system/data/manual_upload_queue.json')
if manual_q.exists():
    items = json.loads(manual_q.read_text())
    if items:
        print('
Manual upload queue entries:', len(items))
        for item in items[-3:]:
            print('-', item.get('niche','?'), '|', item.get('youtube_title','?'))
            print('  file:', item.get('video_path','?'))